## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH08/ch08_Qwen3.ipynb)                                             

# About this notebook

This notebook shows how to run **Qwen3 0.6B** locally with Hugging Face Transformers and control its **reasoning mode** using the chat template. It demonstrates a two phase generation loop with a **thinking budget**, graceful early stop, and safe parsing of the model's hidden thinking and final answer.

## What it shows

* **Model setup** with `AutoModelForCausalLM` and `AutoTokenizer` for `Qwen/Qwen3-0.6B`, using `device_map="auto"` and `torch_dtype="auto"`.
* **Reasoning mode toggle** via `tokenizer.apply_chat_template(..., enable_thinking=True)`.
* **Thinking budget**: first generate up to `thinking_budget` tokens to let the model think privately.
* **Early stop and handoff**: if thinking is still ongoing at the budget, append a short handoff string that closes thinking and prompts the model to answer.
* **Two phase generation**:

  1. Phase 1: generate hidden thinking up to the budget
  2. Phase 2: continue to the final answer, with proper attention masks.
* **Parsing outputs** by locating special markers:

  * `151668` corresponds to `</think>`
  * `151645` corresponds to `<|im_end|>`
* **Clean separation** of

  * `thinking_content` (private chain)
  * `content` (final user facing answer).

## How it works

1. If the model has not closed thinking or finished the message:

   * Append a short early stopping segment that closes `</think>` and asks to answer now.
   * Concatenate ids and continue generation within the overall `max_new_tokens` cap.
2. Split the result at `</think>` and decode both parts for inspection and logging.

## Why this pattern

* Keeps long internal reasoning bounded by a **budget** to protect latency and cost.
* Preserves model quality by allowing some internal reasoning before producing the final answer.
* Provides deterministic guard rails for workflows that must finish within strict limits.

## Extend and adapt

* Tune `thinking_budget` and `max_new_tokens` for your latency and memory targets.
* Change the early stopping text to fit your application voice.
* Wrap this logic in a function and reuse it across tasks.
* Add safety checks for extremely long contexts or multi turn chats.

## Requirements and notes

* You need a GPU with sufficient memory for `Qwen/Qwen3-0.6B`.
* `max_new_tokens=4096` is used here to keep generation time and KV-cache growth reasonable on Colab.
* Special token ids are obtained from the tokenizer instead of being hard-coded.
* Generation parameters are the defaults from `generate` except where overridden.


# Imports

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model and tokenizer

In [ ]:
model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

# Switching between thinking and non-thinking mode with Transformers from HF

In [ ]:
# prepare the model input
prompt = "What are reasoning models?"
messages = [
    {"role": "user", "content": prompt},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
)

# Set thinking budget


In [ ]:
thinking_budget = 1024
max_new_tokens = 4096

# prepare the model input
prompt = "What are reasoning models, in terms of Large Language Models?"
messages = [
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
input_length = model_inputs.input_ids.size(-1)

# Resolve special token ids from the tokenizer instead of hard-coding them.
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
think_end_id = tokenizer.convert_tokens_to_ids("</think>")

# first generation until thinking budget
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=thinking_budget
)
output_ids = generated_ids[0][input_length:].tolist()

# check if the generation has already finished
if im_end_id not in output_ids:
    # check if the thinking process has finished and prepare the second model input
    if think_end_id not in output_ids:
        print("thinking budget is reached")
        early_stopping_text = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
        early_stopping_ids = tokenizer([early_stopping_text], return_tensors="pt", return_attention_mask=False).input_ids.to(model.device)
        input_ids = torch.cat([generated_ids, early_stopping_ids], dim=-1)
    else:
        input_ids = generated_ids
    attention_mask = torch.ones_like(input_ids, dtype=torch.int64)

    # second generation
    generated_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=input_length + max_new_tokens - input_ids.size(-1)  # could be negative if max_new_tokens is not large enough (early stopping text is 24 tokens)
    )
    output_ids = generated_ids[0][input_length:].tolist()

# parse thinking content
try:
    # rindex finding </think>
    index = len(output_ids) - output_ids[::-1].index(think_end_id)
except ValueError:
    index = 0

# Run inference and print result

In [ ]:
thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

thinking content: <think>
Okay, so the user is asking about reasoning models in the context of Large Language Models (LLMs). Let me start by breaking down what they mean. Reasoning models are typically used for tasks like reasoning, problem-solving, or understanding complex information. They might involve something like a neural network that can process information and make decisions based on that.

Now, considering that they mentioned Large Language Models, I should connect reasoning models to those. LLMs are big models, so reasoning could be part of their capabilities. But how exactly?

Maybe they're referring to models that can reason by analyzing input text. For example, an LLM that can generate a coherent answer to a question by understanding the context. That's a common use case. But what about other types of reasoning models? Like, maybe something that uses a different approach, such as a graph-based model or a specific architecture?

Wait, the user might be confused between dif